In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
from xgboost import XGBClassifier
import numpy as np

In [2]:
df = pd.read_csv("C:/Users/alyss/Downloads/OH260225190059543S845PK/CrashStatistics.csv")
df.drop(columns=['LocalReportNumber','DocumentNumber','HitSkip', 'SecondaryCrash','UnitInError','County','FIPSPlaceCode', 'PhotosTaken', 'OH2', 'OH3','OH1P','OHOther','PrivateProperty','ReportingAgencyNCIC','Narrative','ReportTakenBy','Supplement','CrashReportedDateTime','DispatchedDateTime','ArrivedDateTime','SceneClearedDateTime','OtherInvestigationTime','OfficerName','OfficerBadgeNumber','CheckedByOfficerName','CheckedByBadgeNumber'], inplace=True)

C:\Users\alyss\AppData\Local\Temp\ipykernel_38424\1990946857.py:1: DtypeWarning: Columns (0: OfficerBadgeNumber, 1: CheckedByBadgeNumber) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("C:/Users/alyss/Downloads/OH260225190059543S845PK/CrashStatistics.csv")


In [3]:
df["CrashDateTime"] = pd.to_datetime(df["CrashDateTime"])
df["Year"] = df["CrashDateTime"].dt.year
df["Month"] = df["CrashDateTime"].dt.month
df["Day"] = df["CrashDateTime"].dt.day
df["Hour"] = df["CrashDateTime"].dt.hour
df["Minute"] = df["CrashDateTime"].dt.minute
df = df.drop(columns=["CrashDateTime"])

In [4]:
le = LabelEncoder()
df['CrashSeverity'] = le.fit_transform(df['CrashSeverity'])
print(list(df))
print(df.shape[1])

['CrashSeverity', 'LocalInformation', 'NumberOfUnits', 'InCityVillageTownship', 'CityVillageTownshipName', 'Latitude', 'Longitude', 'LocationRouteType', 'LocationRouteNumber', 'LocationPrefix', 'LocationRoadName', 'LocationRoadType', 'DistanceFromReference', 'DistanceReferenceMeasurement', 'DirectionFromReference', 'ReferenceRouteType', 'ReferenceRouteNumber', 'ReferencePrefix', 'ReferenceName', 'ReferencePointUsed', 'ReferenceRoadType', 'IntersectionOrApproachRelated', 'NumberOfApproaches', 'WithinInterchangeArea', 'LocationFirstHarmfulEvent', 'MannerOfCollision', 'Weather', 'LightCondition', 'ActiveSchoolZoneRelated', 'WorkZoneRelated', 'WorkersPresent', 'LawEnforcementPresentInWorkZone', 'WorkZoneType', 'WorkZoneLocation', 'TotalTimeRoadwayClosed', 'RoadwayDivided', 'DividedLaneTravelDirection', 'DividedMedianType', 'RoadContour', 'RoadCondition', 'RoadSurface', 'TotalInjured', 'TotalKilled', 'TotalMinutes', 'AnimalRelated', 'AnimalDeerRelated', 'AlcoholRelated', 'DrugRelated', 'Bic

In [5]:
# List of feature and target columns
x = df[['Year', 'Month', 'Day', 'Hour', 'Minute', 'NumberOfUnits','LightCondition','Weather','MannerOfCollision', 'RoadwayDivided', "IntersectionOrApproachRelated","NumberOfApproaches","WithinInterchangeArea","Latitude","Longitude"]]
y = df['CrashSeverity']

In [6]:
#Split testing and training data
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)


In [ ]:
#Separate numeric and categorical features
numeric_features = x.select_dtypes(include="number").columns
categorical_features = x.select_dtypes(exclude="number").columns
print(numeric_features)
print(categorical_features)

Index(['Year', 'Month', 'Day', 'Hour', 'Minute', 'NumberOfUnits',
       'NumberOfApproaches', 'Latitude', 'Longitude'],
      dtype='str')
Index(['LightCondition', 'Weather', 'MannerOfCollision', 'RoadwayDivided',
       'IntersectionOrApproachRelated', 'WithinInterchangeArea'],
      dtype='str')


In [17]:
from sklearn.pipeline import Pipeline
import sklearn.preprocessing as pre
from sklearn.compose import ColumnTransformer

#("imputer", SimpleImputer(strategy="median")),
data_transformer = ColumnTransformer(
  transformers = [
    ('rescale numeric', pre.StandardScaler(), numeric_features),
    ('recode categorical', 
      pre.OneHotEncoder(handle_unknown = 'ignore'), 
      categorical_features)
    ])

transformed_data = data_transformer.fit_transform(X_train)
print(transformed_data)

[[-0.05540677  0.95624186 -0.17595816 ...  1.          1.
   0.        ]
 [ 0.64040764  0.39302756 -0.85722443 ...  1.          1.
   0.        ]
 [-0.05540677 -0.4517939  -1.5384907  ...  0.          1.
   0.        ]
 ...
 [-0.05540677  1.51945617  0.73239687 ...  0.          1.
   0.        ]
 [-1.44703559 -0.73340106  1.6407519  ...  0.          1.
   0.        ]
 [ 0.64040764  1.51945617 -0.17595816 ...  1.          1.
   0.        ]]


In [ ]:
#Feature selection
from sklearn.feature_selection import SelectFromModel
feature_selector = SelectFromModel(
    estimator=XGBClassifier(
        n_estimators=200,
        eval_metric="logloss",
        random_state=42
    ),
    threshold="median"
)

In [ ]:
clf = XGBClassifier(n_estimators=100, random_state=42)
clf = clf.fit(transformed_data, y_train)

#print(f"Feature importances: {clf.feature_importances_}")

# 3. Use SelectFromModel to select features based on the trained estimator
# By default, the threshold is the mean of the feature importances
model_selection = SelectFromModel(clf, prefit=True) 

X_train_selected = model_selection.transform(transformed_data)
print(X_train_selected)

selected_feature_indices = model_selection.get_support(indices=True)
selected_feature_names = [X_train_selected[i] for i in selected_feature_indices]
#print("Selected feature names:", selected_feature_names)


[[-0.05540677  0.44683659 -0.24379114 ...  0.          0.
   1.        ]
 [ 0.64040764  0.44683659 -0.24379114 ...  0.          0.
   1.        ]
 [-0.05540677 -0.81187658         nan ...  0.          0.
   1.        ]
 ...
 [-0.05540677  0.44683659         nan ...  0.          0.
   1.        ]
 [-1.44703559  0.44683659         nan ...  0.          0.
   0.        ]
 [ 0.64040764 -0.81187658 -0.24379114 ...  0.          0.
   1.        ]]


: 